# Hybrid ingestion trên Wikimedia EventStreams: stream, batch và đối soát

Mọi sửa đổi trên Wikipedia, Wikidata, Commons... (hơn 900 wiki) được Wikimedia phát công khai, theo
thời gian thực, qua **EventStreams**: một endpoint HTTP dùng giao thức **Server-Sent Events** (SSE),
phía sau là Kafka. Luồng `recentchange` có khoảng 30–50 event mỗi giây.

Bài này dựng một pipeline **hybrid** thu nhỏ trên cùng một nguồn:

```text
EventStreams ──record()──> data/raw/wikimedia-recentchange/<thời điểm>.jsonl   (lớp raw, bất biến)
                                   │
                ┌──────────────────┴──────────────────┐
     phần stream: stream_counts()          phần batch: build_clean() + batch_counts()
     một lượt, theo thứ tự đến,            đọc hết tệp, dedup toàn bộ,
     watermark + dedup có TTL              group by theo event time
     (nhanh, có thể thiếu/dư)              (chậm, đúng)
                └──────────────────┬──────────────────┘
                            reconcile(): chênh lệch theo (phút, wiki)
```

Stream trực tiếp không tái lập được, nên ta **ghi lại một đoạn** trước, rồi mọi phân tích và chấm
điểm chạy trên tệp đã ghi. Đó cũng là cách lớp raw hoạt động ngoài đời: ghi nguyên bản, xử lý sau.

Cell gắn tag `explore` (dòng đầu `# explore: ...`) ghi stream, đọc dữ liệu thật và vẽ; khi chấm chúng
bị bỏ qua, test gọi thẳng các hàm bạn viết (trên fixture nhỏ, và trên tệp bạn đã ghi nếu có). Chấm:
`uv run pytest tracks/data-engineer/modules/07-streaming-kafka/exercises/advanced/03-hybrid-wikimedia -q`

In [ ]:
# explore: tìm gốc repo (thư mục có pyproject.toml), chuẩn bị thư mục raw/clean
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

RAW = ROOT / "data" / "raw" / "wikimedia-recentchange"
CLEAN = ROOT / "data" / "clean" / "wikimedia-recentchange"
RAW.mkdir(parents=True, exist_ok=True)
print(RAW, sorted(p.name for p in RAW.glob("*.jsonl")))

In [ ]:
import json
import time
from collections.abc import Iterable, Iterator
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

STREAM_URL = "https://stream.wikimedia.org/v2/stream/recentchange"
# Chính sách của Wikimedia: User-Agent phải cho biết ai gọi và cách liên hệ. Sửa phần contact.
USER_AGENT = "de-track-learner/1.0 (hybrid ingestion exercise; contact: your-email@example.com)"
EDIT_TYPES = ("edit", "new")   # sửa trang và tạo trang mới; categorize/log không tính là "sửa"

## 1. Ghi stream: SSE bằng `requests`, không cần thư viện riêng

SSE là văn bản thuần, mỗi event là vài dòng `trường: giá trị`, kết thúc bằng **một dòng trống**:

```text
:ok                                   <- dòng bắt đầu bằng ':' là comment (heartbeat), bỏ qua
event: message
id: [{"topic":"codfw.mediawiki.recentchange","partition":0,"timestamp":1793255168093}]
data: {"$schema":"/mediawiki/recentchange/1.0.0","meta":{...},"type":"edit",...}
                                      <- dòng trống: phát event
```

`iter_sse(lines)` nhận các dòng (đã bỏ hoặc chưa bỏ `\n`), trả `(last_event_id, data)`:

- bỏ dòng comment `:`; tách `trường:giá trị`, bỏ **một** dấu cách đầu giá trị;
- nhiều dòng `data:` của cùng một event nối bằng `"\n"`;
- `id` là "last event id": giữ nguyên cho các event sau nếu chúng không có `id` riêng;
- event chỉ phát khi gặp dòng trống. Event dở dang khi luồng đứt **không** được phát (nó chưa đủ).

`record(seconds, out_path, *, session=None, max_events=None, retry_wait=1.0, max_retries=5) -> dict`:

- `session.get(STREAM_URL, headers=..., stream=True, timeout=...)` (mặc định `requests.Session()`),
  header `User-Agent: USER_AGENT` (chính sách Wikimedia: tên công cụ + cách liên hệ, **sửa email trong
  `USER_AGENT`**) và `Accept: text/event-stream`;
- mỗi event ghi **một dòng** JSONL `{"received_at": "2026-10-29T06:26:08.123Z", "event": {...nguyên bản...}}`.
  `received_at` là giờ UTC lúc nhận, độ chính xác mili giây: đây là **ingestion time**, cần để đo độ trễ;
- đứt kết nối (`requests.RequestException`) hoặc server trả 429/5xx: đợi `retry_wait` (lùi dần) rồi nối
  lại với header `Last-Event-ID` = id của event cuối đã ghi, để không mất đoạn giữa (đổi lại có thể
  nhận **trùng**: đó là at-least-once). Lỗi 4xx khác thì ném lỗi ra ngay. Quá `max_retries` lần liên tiếp thì dừng;
- ghi vào `out_path` + `.part`, xong mới đổi tên thành `out_path`: tệp `.jsonl` luôn là tệp hoàn chỉnh;
- trả `{"path": str(out_path), "events": số event đã ghi, "reconnects": số lần nối lại}`.

In [ ]:
def iter_sse(lines: Iterable[str]) -> Iterator[tuple[str | None, str]]:
    """Dòng SSE -> (last event id, data). Event dở dang ở cuối luồng bị bỏ."""
    # TODO
    raise NotImplementedError


def record(seconds: float, out_path: Path, *, session=None, max_events: int | None = None,
           retry_wait: float = 1.0, max_retries: int = 5) -> dict:
    """Ghi stream ra JSONL {"received_at", "event"}; nối lại với Last-Event-ID; ghi qua tệp .part."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: ghi vài phút stream thật (chỉ ghi khi chưa có tệp nào; đặt RECORD_NEW = True để ghi thêm)
from datetime import datetime, timezone

RECORD_SECONDS = 300
RECORD_NEW = not any(RAW.glob("*.jsonl"))
if RECORD_NEW:
    out = RAW / f"{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}.jsonl"
    print("Đang ghi", RECORD_SECONDS, "giây vào", out)
    print(record(RECORD_SECONDS, out))

FILES = sorted(RAW.glob("*.jsonl"))
LINES = [line for path in FILES for line in path.open(encoding="utf-8")]
print(len(FILES), "tệp,", f"{len(LINES):,} dòng,", f"{sum(p.stat().st_size for p in FILES) / 1e6:.1f} MB")

## 2. Đọc một event đã ghi

`parse_event(line) -> dict | None` trả dict phẳng với đúng các khoá:

| Khoá | Lấy từ | Ghi chú |
| --- | --- | --- |
| `event_id` | `event.meta.id` | UUID của event, **khoá dedup** (khác `event.id` là id của dòng recent change trong wiki) |
| `wiki`, `domain` | `event.wiki`, `event.meta.domain` | `enwiki`, `en.wikipedia.org` |
| `type` | `event.type` | `edit`, `new`, `categorize`, `log`... |
| `bot`, `namespace`, `title`, `user` | cùng tên | `bot` là `bool`; `namespace` là `int`; `user` có thể thiếu thì `None` |
| `event_time` | `event.timestamp` | `int` giây Unix: **lúc sửa đổi xảy ra** (event time) |
| `received_at` | `received_at` | `float` giây Unix: lúc ta nhận (ingestion time) |
| `len_delta` | `length.new - length.old` | trang mới không có `old` thì coi là 0; không có `length` thì `None` |

Trả `None` (không ném lỗi) khi: dòng không phải JSON hoặc ghi dở, thiếu một trong `meta.id`, `type`,
`wiki`, `timestamp`, `received_at`; hoặc là **canary event** (`meta.domain == "canary"`): Wikimedia tự
bắn event giả định kỳ để giám sát luồng, không phải sửa đổi thật.

In [ ]:
def parse_event(line: str) -> dict | None:
    """Một dòng JSONL đã ghi -> dict phẳng theo bảng trên, hoặc None."""
    # TODO
    raise NotImplementedError

## 3. Phần stream: một lượt, theo thứ tự đến

`stream_counts(events, window=60, allowed_lateness=10, dedup=True, dedup_ttl=600)` giả lập một stream
processor (như Flink/Spark Structured Streaming) đếm **số sửa đổi** (`type` thuộc `EDIT_TYPES`) theo
tumbling window event time `window` giây và theo `wiki`. Duyệt `events` **đúng thứ tự đã cho** (thứ tự
đến), không sắp xếp, không sửa list đầu vào. Với mỗi event sửa đổi, theo đúng thứ tự:

1. `stats["events"] += 1`;
2. nếu `dedup`: id đã có trong bộ nhớ thì `duplicates += 1`, bỏ qua; chưa có thì nhớ `id -> event_time`;
3. **late**: `window_end = start + window` với `start = t - t % window`. Nếu đã thấy event trước đó và
   `window_end <= max_event_time - allowed_lateness` (watermark tính **trước** khi cập nhật) thì
   `late += 1`, bỏ qua;
4. nếu `t > max_event_time`: cập nhật, và (nếu `dedup`) quên các id có `event_time < max_event_time - dedup_ttl`;
5. cộng 1 vào `(start, wiki)`, `counted += 1`.

Trả `(DataFrame[window_start, wiki, edits], stats)`: `window_start` là `datetime64` UTC, `edits` số nguyên,
sắp theo `(window_start, wiki)`; `stats = {"events", "duplicates", "late", "counted"}`.

Vì sao nhớ id **trước** khi xét late? Để bản trùng của một event trễ được đếm là trùng, không bị đếm
late hai lần; nhờ vậy `tổng batch - tổng stream == late` khi TTL đủ dài. Ở đây kết quả của mọi window
được "phát" một lần ở cuối; processor thật phát khi watermark vượt `window_end`.

In [ ]:
def stream_counts(events: Iterable[dict], window: int = 60, allowed_lateness: int = 10,
                  dedup: bool = True, dedup_ttl: int = 600) -> tuple[pd.DataFrame, dict]:
    """Đếm sửa đổi theo (window, wiki) như một stream processor: một lượt, theo thứ tự đến."""
    # TODO
    raise NotImplementedError

## 4. Phần batch: đọc hết rồi mới tính

`build_clean(lines) -> (df, stats)`:

- `parse_event` từng dòng; `None` thì tính vào `dropped`;
- dedup theo `event_id`, **giữ bản đến đầu tiên** (thứ tự dòng trong tệp);
- `event_time`, `received_at` thành `datetime64` UTC; thêm `lag_s = received_at - event_time` (giây,
  `float`); `bot` kiểu `bool`, `namespace` `int64`, `len_delta` `Int64` (cho phép thiếu);
- sắp theo `(event_time, event_id)`;
- `stats = {"lines", "parsed", "dropped", "duplicates"}` với `parsed` là số event đọc được **trước** dedup.

`batch_counts(df, window=60)`: cùng định nghĩa với phần stream (`EDIT_TYPES`, window theo event time),
nhưng trên toàn bộ dữ liệu đã dedup. Cùng cột `[window_start, wiki, edits]`, cùng thứ tự sắp.

In [ ]:
def build_clean(lines: Iterable[str]) -> tuple[pd.DataFrame, dict]:
    """Batch: đọc hết, bỏ dòng hỏng/canary, dedup theo event_id (giữ bản đến đầu), ép kiểu."""
    # TODO
    raise NotImplementedError


def batch_counts(df: pd.DataFrame, window: int = 60) -> pd.DataFrame:
    """Số sửa đổi (EDIT_TYPES) theo (window event time, wiki) trên dữ liệu sạch."""
    # TODO
    raise NotImplementedError

## 5. Đối soát stream với batch

`reconcile(stream, batch) -> DataFrame` với cột `[window_start, wiki, stream_edits, batch_edits, diff]`:
outer join theo `(window_start, wiki)`, bên nào thiếu khoá thì là 0 (số nguyên), `diff = batch_edits -
stream_edits`, sắp theo `(window_start, wiki)`.

- `diff > 0`: stream **thiếu** (event trễ bị bỏ);
- `diff < 0`: stream **dư** (trùng mà dedup không bắt được: tắt dedup, hoặc TTL ngắn hơn khoảng cách giữa hai bản);
- tổng khớp chưa chắc từng window khớp: thiếu ở window này và dư ở window kia triệt tiêu nhau. Đối soát theo khoá.

In [ ]:
def reconcile(stream: pd.DataFrame, batch: pd.DataFrame) -> pd.DataFrame:
    """Outer join theo (window_start, wiki); thiếu bên nào thì 0; diff = batch - stream."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: chạy cả hai phần trên dữ liệu đã ghi, đối soát, và thử nhiều mức allowed_lateness
EVENTS = [e for e in map(parse_event, LINES) if e is not None]
df, STATS = build_clean(LINES)
BATCH = batch_counts(df)
STREAM, SSTATS = stream_counts(EVENTS, allowed_lateness=10)
RECON = reconcile(STREAM, BATCH)
print("build_clean:", STATS)
print("stream (lateness 10s):", SSTATS)
print("window lệch:", int((RECON["diff"] != 0).sum()), "/", len(RECON), "- tổng diff:", int(RECON["diff"].sum()))

LATENESS = [0, 1, 2, 5, 10, 20, 30, 60, 120]
LATE = pd.DataFrame([{"allowed_lateness": a, **stream_counts(EVENTS, allowed_lateness=a)[1]} for a in LATENESS])
LATE["late_rate"] = LATE["late"] / LATE["events"]
LATE

## 6. Insight

`insights(df, stats) -> dict` (dùng `df, stats` từ `build_clean`) với đúng các khoá:

| Khoá | Nghĩa |
| --- | --- |
| `events`, `duplicates`, `duplicate_rate`, `dropped` | số event sau dedup, số bản trùng, `duplicates / parsed`, số dòng bị bỏ |
| `duration_min` | `(max received_at - min received_at)` tính bằng phút: thời lượng ghi thật |
| `edits`, `edits_per_min` | số sửa đổi (`EDIT_TYPES`) và `edits / duration_min` |
| `bot_share` | tỉ lệ sửa đổi do bot (`bot == True`) trong các sửa đổi |
| `wikis` | số wiki khác nhau (mọi loại event) |
| `top_wiki`, `top_wiki_edits` | wiki nhiều sửa đổi nhất và số sửa đổi; hoà thì lấy tên nhỏ hơn theo thứ tự chữ |
| `top_human_wiki` | wiki nhiều sửa đổi **của người** nhất (cùng quy tắc hoà) |
| `type_counts` | `dict` `type -> số event` (mọi loại) |
| `lag_p50_s`, `lag_p95_s`, `lag_max_s` | trung vị, phân vị 95 (`quantile`, nội suy tuyến tính) và max của `lag_s` |

In [ ]:
def insights(df: pd.DataFrame, stats: dict) -> dict:
    """Các con số trong bảng trên."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: các con số để viết report.md
INSIGHTS = insights(df, STATS)
INSIGHTS

## 7. Biểu đồ

Vẽ ít nhất ba biểu đồ (mỗi biểu đồ một cell `explore`):

1. số sửa đổi **mỗi phút**, tách bot và người (stacked bar): nhịp có đều không, bot chiếm bao nhiêu?
2. **top 10 wiki** theo số sửa đổi, tách bot và người: wiki nào gần như toàn bot?
3. **độ trễ** `lag_s` (histogram, trục log) kèm đường **tỉ lệ late theo `allowed_lateness`** (bảng `LATE`):
   chọn `allowed_lateness` bao nhiêu thì mất dưới 1% event?
4. (tuỳ chọn) đối soát theo phút: tổng `stream_edits` và `batch_edits` mỗi phút, hoặc heatmap `diff`.

In [ ]:
# explore: sửa đổi mỗi phút, bot vs người
# TODO: stacked bar theo phút (event time), một màu bot, một màu người

In [ ]:
# explore: top 10 wiki
# TODO: barh top 10 wiki theo số sửa đổi, tách bot và người

In [ ]:
# explore: độ trễ và tỉ lệ late
# TODO: histogram lag_s (trục log) + đường late_rate theo allowed_lateness từ bảng LATE

In [ ]:
# explore: (tuỳ chọn) đối soát theo phút

## 8. Ghi dữ liệu sạch

`write_clean(df, out_dir)` ghi **một** tệp `out_dir/recentchange.parquet` (ghi đè, tạo thư mục nếu
chưa có) qua tệp tạm rồi `replace`, trả đường dẫn. Chạy lại không nhân đôi: pipeline **idempotent**.

In [ ]:
def write_clean(df: pd.DataFrame, out_dir: Path) -> Path:
    """Ghi đè out_dir/recentchange.parquet qua tệp tạm; trả đường dẫn."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: ghi dữ liệu sạch
path = write_clean(df, CLEAN)
print(path, f"{path.stat().st_size / 1e6:.1f} MB", len(pd.read_parquet(path)), "dòng")

## 9. Báo cáo và câu hỏi thiết kế

Viết `report.md` cạnh notebook (4 mục **Bộ dữ liệu**, **Làm sạch**, **Insight** có số, **Giới hạn**).
Trong mục Insight, trả lời bằng số đo trên chính tệp bạn ghi:

- tỉ lệ sửa đổi của bot, wiki sôi động nhất, số sửa đổi mỗi phút;
- `lag_p50_s`, `lag_p95_s`; với `allowed_lateness = 10` giây thì stream bỏ bao nhiêu % sửa đổi, cần bao nhiêu
  giây để dưới 1%;
- có bản trùng nào không, và nếu không thì vì sao vẫn phải dedup.

Câu hỏi thiết kế (viết vào mục Giới hạn hoặc một mục thêm): nếu phải làm dashboard "sửa đổi mỗi phút
theo wiki" cho Wikimedia, bạn chọn **Lambda** (stream cho số tạm + batch sửa lại) hay **Kappa** (chỉ stream,
replay khi cần tính lại)? EventStreams chỉ cho replay khoảng vài ngày (`since`/`Last-Event-ID`): điều
đó ảnh hưởng thế nào tới lựa chọn?